In [1]:
# Install required libraries if not installed yet
# !pip install transformers torch

In [2]:
import os
import warnings
import logging

# Suppress non-critical warnings and unnecessary logs
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.ERROR, force=True)
logging.getLogger("httpx").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

import torch
from transformers import pipeline, GenerationConfig
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

MODEL_NAME = "gpt2"
PROMPT = "In the future, artificial intelligence will"

PIPELINE_IO_CONFIG = {
    "truncation": True,
    "clean_up_tokenization_spaces": False,
}

In [3]:
def resolve_device() -> int:
    """
    Detect available hardware.
    Returns 0 if CUDA GPU is available, otherwise -1 for CPU.
    """
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"Using device: GPU (CUDA) - {gpu_name}")
        return 0

    print("Using device: CPU")
    return -1


def build_generator(model_name: str = MODEL_NAME):
    """Build the text-generation pipeline using the selected device."""
    device = resolve_device()

    print("Loading GPT-2 text generation model...")
    generator = pipeline(
        "text-generation",
        model=model_name,
        device=device
    )

    # GPT-2 has no default pad token, so eos_token_id is used as pad_token_id
    generator.tokenizer.pad_token_id = generator.tokenizer.eos_token_id

    print("Model loaded successfully!")

    return generator


def build_generation_config(tokenizer) -> GenerationConfig:
    """Set the text generation parameters."""
    return GenerationConfig(
        max_new_tokens=50,
        num_return_sequences=2,
        do_sample=True,
        temperature=0.75,
        top_k=50,
        top_p=0.92,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
    )


def generate(generator, prompt: str, generation_config: GenerationConfig):
    """Generate text using the selected model and generation settings."""
    return generator(
        prompt,
        generation_config=generation_config,
        **PIPELINE_IO_CONFIG,
    )


def display_results(prompt: str, results) -> None:
    """Display the prompt and generated text clearly."""
    divider = "\n---\n"

    print(divider)
    print("## Prompt")
    print(prompt.strip())
    print(divider)

    for i, sequence in enumerate(results, start=1):
        print(f"### Generated Text {i}")
        print(sequence["generated_text"].strip())
        print(divider)

In [4]:
generator = build_generator()
generation_config = build_generation_config(generator.tokenizer)
results = generate(generator, PROMPT, generation_config)
display_results(PROMPT, results)

Using device: CPU
Loading GPT-2 text generation model...
Model loaded successfully!

---

## Prompt
In the future, artificial intelligence will

---

### Generated Text 1
In the future, artificial intelligence will be able to do things like create more accurate information about a person's physical location. In this case there would also be no need for doctors or medical staff and you could simply take your own time with it all instead of being forced into doing so by

---

### Generated Text 2
In the future, artificial intelligence will be able to predict what's going on in your life and how you're feeling.
"If we don't have any way of knowing about that now," says Dr Jangong from China Academy for Information Sciences (ACIS), "we can

---

